# 🚀 Learning RAGAS (Retrieval Augmented Generation Assessment)

RAGAS is a framework that helps you evaluate your RAG pipelines. Instead of manually checking answers, RAGAS uses an **LLM-as-a-judge** to automatically score your system.

In this tutorial, we use the **Gemini Free Tier** as our judge!

### Important: Ragas v0.4.3 has two metric systems
| System | Import path | LLM type required | Evaluation method |
|--------|------------|-------------------|-------------------|
| Legacy | `ragas.metrics` | `LangchainLLMWrapper` | `evaluate()` |
| **Collections (new)** | `ragas.metrics.collections` | `InstructorBaseRagasLLM` via `llm_factory` | `await .ascore()` |

We'll use the **new collections** system in this tutorial.

## 1. Installation

In [1]:
!pip install ragas python-dotenv openai


[notice] A new release of pip is available: 26.0.1 -> 26.1
[notice] To update, run: python.exe -m pip install --upgrade pip


## 2. Load Environment Variables

In [2]:
import os
from dotenv import load_dotenv

load_dotenv()

if not os.environ.get("GEMINI_API_KEY"):
    print("⚠️ Warning: GEMINI_API_KEY not found in .env file!")
else:
    print("✅ GEMINI_API_KEY loaded successfully!")

✅ GEMINI_API_KEY loaded successfully!


## 3. Set up the Judge LLM and Embeddings
We use `llm_factory` with Gemini's **OpenAI-compatible endpoint** to create the `InstructorBaseRagasLLM` that collections metrics require.

In [3]:
from openai import OpenAI
from ragas.llms import llm_factory
from ragas.embeddings import embedding_factory

print("🔧 Setting up Judge LLM and Embeddings...")

# Gemini exposes an OpenAI-compatible REST endpoint
GEMINI_BASE_URL = 'https://generativelanguage.googleapis.com/v1beta/openai/'

gemini_client = OpenAI(
    api_key=os.environ['GEMINI_API_KEY'],
    base_url=GEMINI_BASE_URL
)

judge_llm = llm_factory('gemini-2.0-flash', client=gemini_client)
judge_embeddings = embedding_factory('openai', client=gemini_client, model='text-embedding-004')

print("✅ Setup complete!")

c:\Users\sujat\projects\AI\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
c:\Users\sujat\projects\AI\.venv\Lib\site-packages\instructor\providers\gemini\client.py:5: FutureWarning: 

All support for the `google.generativeai` package has ended. It will no longer be receiving 
updates or bug fixes. Please switch to the `google.genai` package as soon as possible.
See README for more details:

https://github.com/google-gemini/deprecated-generative-ai-python/blob/main/README.md

  import google.generativeai as genai  # type: ignore[import-not-found]


🔧 Setting up Judge LLM and Embeddings...


C:\Users\sujat\AppData\Local\Temp\ipykernel_15492\2123916432.py:16: DeprecationWarning: Importing embedding_factory from ragas.embeddings is deprecated. Import directly from ragas.embeddings.base or use modern providers: from ragas.embeddings import OpenAIEmbeddings, GoogleEmbeddings, HuggingFaceEmbeddings
  judge_embeddings = embedding_factory('openai', client=gemini_client, model='text-embedding-004')


✅ Setup complete!


## 4. Understanding RAGAS Metrics

| Metric | What it measures | Needs `reference`? |
|--------|-----------------|--------------------|
| **Faithfulness** | Is the answer grounded in the context? (catches hallucinations) | No |
| **Answer Relevancy** | Does the answer address the question? | No |
| **Context Precision** | Are the most relevant chunks ranked at the top? | Yes |
| **Context Recall** | Did the retriever find all necessary info? | Yes |

In [4]:
from ragas.metrics.collections import (
    Faithfulness,
    AnswerRelevancy,
    ContextPrecision,
    ContextRecall
)

faithfulness = Faithfulness(llm=judge_llm)
answer_relevancy = AnswerRelevancy(llm=judge_llm, embeddings=judge_embeddings)
context_precision = ContextPrecision(llm=judge_llm)
context_recall = ContextRecall(llm=judge_llm)

print("✅ Metrics initialized!")

✅ Metrics initialized!


## 5. Prepare Test Data
Collections metrics use `await metric.ascore()` with keyword arguments:
- `user_input` — the question
- `response` — the RAG system's answer
- `retrieved_contexts` — list of retrieved text chunks
- `reference` — ground truth answer (for precision/recall metrics)

> **Why `ascore` instead of `score`?** Jupyter notebooks run inside an async event loop, so synchronous `.score()` raises a RuntimeError. We use `await .ascore()` instead.

In [5]:
# Sample 1: Good answer — faithful to context
sample_good = {
    "user_input": "Who wrote the Attention is All You Need paper?",
    "response": "The paper was authored by Ashish Vaswani, Noam Shazeer, Niki Parmar, Jakob Uszkoreit, Llion Jones, Aidan N. Gomez, Łukasz Kaiser, and Illia Polosukhin.",
    "retrieved_contexts": [
        "Attention Is All You Need. Ashish Vaswani, Google Brain. Noam Shazeer, Google Brain. Niki Parmar, Google Research. Jakob Uszkoreit, Google Research.",
        "Llion Jones, Google Research. Aidan N. Gomez, University of Toronto. Łukasz Kaiser, Google Brain. Illia Polosukhin."
    ],
    "reference": "Ashish Vaswani, Noam Shazeer, Niki Parmar, Jakob Uszkoreit, Llion Jones, Aidan N. Gomez, Łukasz Kaiser, and Illia Polosukhin."
}

# Sample 2: Bad answer — contains a HALLUCINATION about Jupiter
sample_hallucinated = {
    "user_input": "What is the capital of France?",
    "response": "Paris is the capital of France, and Jupiter has 95 moons.",
    "retrieved_contexts": [
        "France is a country in Western Europe. Its capital is Paris, which is known for the Eiffel Tower."
    ],
    "reference": "Paris is the capital of France."
}

print("✅ Test samples ready!")

✅ Test samples ready!


## 6. Score Individual Samples
We use `await metric.ascore()` since Jupyter runs an async event loop. Each call returns a `MetricResult` with `.value` (0.0–1.0).

In [6]:
import pandas as pd

print("⏳ Scoring... (this takes ~30 seconds on free tier)")
print()

results = []

for label, sample in [("Good Answer", sample_good), ("Hallucinated Answer", sample_hallucinated)]:
    print(f"--- {label} ---")
    
    f_result = await faithfulness.ascore(**sample)
    print(f"  Faithfulness:      {f_result.value:.2f}")
    
    ar_result = await answer_relevancy.ascore(**sample)
    print(f"  Answer Relevancy:  {ar_result.value:.2f}")
    
    cp_result = await context_precision.ascore(**sample)
    print(f"  Context Precision: {cp_result.value:.2f}")
    
    cr_result = await context_recall.ascore(**sample)
    print(f"  Context Recall:    {cr_result.value:.2f}")
    print()
    
    results.append({
        "Sample": label,
        "Faithfulness": f_result.value,
        "Answer Relevancy": ar_result.value,
        "Context Precision": cp_result.value,
        "Context Recall": cr_result.value
    })

df = pd.DataFrame(results)
print("🎉 All Done!")
display(df)

⏳ Scoring... (this takes ~30 seconds on free tier)

--- Good Answer ---


TypeError: Faithfulness.ascore() got an unexpected keyword argument 'reference'

## 7. Batch Scoring
For larger datasets, use `await metric.abatch_score()` which takes a list of input dicts.

In [ ]:
batch_inputs = [sample_good, sample_hallucinated]

print("⏳ Running batch Faithfulness scoring...")
batch_results = await faithfulness.abatch_score(batch_inputs)

for i, r in enumerate(batch_results):
    print(f"  Sample {i+1} — Faithfulness: {r.value:.2f}")

print("\n✅ Batch scoring complete!")

## 8. Analyze the Results

**What to look for:**
- The **Good Answer** should score high (~1.0) on Faithfulness — everything it says is in the context.
- The **Hallucinated Answer** should score low on Faithfulness — Jupiter's moons is NOT in the context.
- Both should have decent Context Recall since the contexts do contain the ground truth info.

**Try experimenting!** Change the `response` or `retrieved_contexts` in Step 5 and re-run to see how scores change.